# Statistical fidelity, spatial fidelity and disclosure-risk assessment


In [ ]:
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    PACKAGE_ROOT = PACKAGE_ROOT.parent
if not (PACKAGE_ROOT / "geoctgan").is_dir():
    raise FileNotFoundError("geoctgan")
os.chdir(PACKAGE_ROOT)
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from geoctgan.config import Config

def input_path(label, required=True):
    raw = input(label + ": ").strip().strip('"').strip("'")
    if not raw:
        if not required:
            return None
        raise ValueError(label)
    path = Path(raw).expanduser().resolve()
    if required and not path.exists():
        raise FileNotFoundError(label)
    return path

def stored_path(path):
    return str(Path(path).resolve())

@contextmanager
def configuration_file(values):
    with TemporaryDirectory(prefix="geoctgan_") as directory:
        path = Path(directory) / "configuration.json"
        path.write_text(json.dumps(values, indent=2) + "\n", encoding="utf-8")
        yield path


## Evaluation inputs


In [ ]:
GENERATION_SUMMARY = input_path("Generation run_summary.json")
DA_BOUNDARIES = input_path("Toronto DA boundaries")
MARGINALIZATION = input_path("ON-Marg table")
BENCHMARK_MANIFEST = input_path("Benchmark manifest CSV", required=False)
OUTPUT_PARENT = input_path("Output directory", required=False) or (PACKAGE_ROOT / "Output")
RUN_ROOT = OUTPUT_PARENT / "Toronto"
MODEL_SEEDS = [42, 52, 62]


## Held-out evaluation


In [ ]:
from geoctgan.evaluation import EvaluationConfig, run_evaluation

evaluation = run_evaluation(
    GENERATION_SUMMARY,
    output_parent=OUTPUT_PARENT,
    evaluation_dir=RUN_ROOT / "Evaluation",
    da_boundaries=DA_BOUNDARIES,
    marginalization_csv=MARGINALIZATION,
    benchmark_manifest=BENCHMARK_MANIFEST,
    config=EvaluationConfig(
        heavy_privacy_datasets=[
            f"GeoCTGAN-{profile} | seed {seed}"
            for profile in ("Full", "Priv", "UpperQuota")
            for seed in MODEL_SEEDS
        ],
        privacy_profile="toronto",
    ),
)
